# Fink/LSST — Alert Statistics on LSSTCam Focal Plane

This notebook reads **all** Parquet files saved by `01_fink_block_lightcurves.ipynb`
in `data_FINK_BLOCK_LC_01/` and produces:

1. A **global focal-plane heatmap** (alert count per CCD, all groups, all bands, all time)
2. A **per-band 2×3 grid** of focal-plane heatmaps (shared log colour scale)
3. A **month-by-month subplot grid** of focal-plane heatmaps showing the temporal
   evolution of alert coverage across the 189-CCD focal plane

**Expected data** in `data_FINK_BLOCK_LC_01/`:
- `{group}_fp.parquet`  — forced-photometry light curves
- `{group}_src.parquet` — detection-based light curves (diaSources)

No API call is made in this notebook.

> **Note on `r:detector`**: this column must be present in the `*_src.parquet` files.
> If it is missing, re-run `01_fink_block_lightcurves.ipynb` after adding `r:detector`
> to `COLS_SRC`.

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/CNRS/IN2P3
- **creation date** : 2026-04-09
- **last update** : 2026-04-09
- **subject:** Fink alert Broker applied to Rubin LSST alerts — focal-plane statistics

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import calendar
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # written by notebook 01
DIR_FIGS = f"figs_{NB_TAG}_10"  # figures specific to this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Focal-plane geometry ──────────────────────────────────────────────────────
# Shared with notebooks 04_calib/05* — same CSV format
GEOMETRY_CSV = "data_FocalPlane/ccd_geometry.csv"

# ── Photometric bands ─────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Plot settings ─────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save current figure to DIR_FIGS in both PDF and PNG formats."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Geometry CSV     : {os.path.abspath(GEOMETRY_CSV)}")

## 2. Load focal-plane geometry

In [ ]:
geom = pd.read_csv(GEOMETRY_CSV)

# Compute radial distance from focal-plane centre [mm]
geom["r_mm"] = np.sqrt(geom["x_center"] ** 2 + geom["y_center"] ** 2)

# Focal-plane bounding box (used for all heatmap axes limits)
fp_xmin = geom[[f"corner{i}_x" for i in range(4)]].min().min()
fp_xmax = geom[[f"corner{i}_x" for i in range(4)]].max().max()
fp_ymin = geom[[f"corner{i}_y" for i in range(4)]].min().min()
fp_ymax = geom[[f"corner{i}_y" for i in range(4)]].max().max()

print(f"Geometry loaded: {len(geom)} CCDs")
print(f"Focal plane X : [{fp_xmin:.1f}, {fp_xmax:.1f}] mm")
print(f"Focal plane Y : [{fp_ymin:.1f}, {fp_ymax:.1f}] mm")
geom.head(3)

## 3. Load all *_src.parquet files and concatenate

All source groups stored in `data_FINK_BLOCK_LC_01/` are loaded.
The `r:detector` column is cast to nullable `Int64`; rows without a valid
detector number are dropped for the focal-plane analysis.

A `month` column (YYYY-MM string) is derived from `r:midpointMjdTai` for
the temporal evolution plots.

In [ ]:
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))
print(f"Found {len(src_files)} *_src.parquet files:")
for f in src_files:
    print(f"  {os.path.basename(f)}")

In [ ]:
def load_src_files(file_list):
    """
    Load and concatenate all *_src.parquet files.

    Returns a DataFrame with at minimum the columns:
      r:detector, r:visit, r:midpointMjdTai, r:band, group
    plus a 'month' column (YYYY-MM) derived from the MJD.
    """
    dfs = []
    missing_detector = []

    for path in file_list:
        df = pd.read_parquet(path)

        # Cast r:detector to nullable integer
        if "r:detector" in df.columns:
            df["r:detector"] = pd.to_numeric(df["r:detector"], errors="coerce").astype("Int64")
        else:
            missing_detector.append(os.path.basename(path))
            df["r:detector"] = pd.NA  # fill with NA for tracking

        # Cast r:visit to nullable integer
        if "r:visit" in df.columns:
            df["r:visit"] = pd.to_numeric(df["r:visit"], errors="coerce").astype("Int64")

        # Cast MJD
        if "r:midpointMjdTai" in df.columns:
            df["r:midpointMjdTai"] = pd.to_numeric(df["r:midpointMjdTai"], errors="coerce")

        # Ensure group column exists (set from filename if absent)
        if "group" not in df.columns:
            group_name = os.path.basename(path).replace("_src.parquet", "")
            df["group"] = group_name

        dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    df_all = pd.concat(dfs, ignore_index=True)

    if missing_detector:
        print(f"WARNING: r:detector missing in {len(missing_detector)} file(s):")
        for f in missing_detector:
            print(f"    {f}")
        print("  → Re-run 01_fink_block_lightcurves.ipynb after adding r:detector to COLS_SRC.")

    # Derive calendar month from MJD (requires astropy or a simple offset)
    # MJD 0 = 1858-11-17; use pandas Timestamp arithmetic
    if "r:midpointMjdTai" in df_all.columns:
        mjd_series = df_all["r:midpointMjdTai"]
        # Convert MJD to pandas Timestamp (MJD epoch: 1858-11-17)
        MJD_EPOCH = pd.Timestamp("1858-11-17")
        df_all["obs_date"] = MJD_EPOCH + pd.to_timedelta(mjd_series, unit="D")
        df_all["month"] = df_all["obs_date"].dt.to_period("M").astype(str)  # YYYY-MM

    return df_all


df_all_src = load_src_files(src_files)

print(f"\nTotal rows loaded (all groups, all bands) : {len(df_all_src):,}")
print(f"Rows with valid r:detector                : {df_all_src['r:detector'].notna().sum():,}")
print(f"Unique groups                             : {df_all_src['group'].nunique()}")
print(f"Unique CCDs (r:detector)                  : {df_all_src['r:detector'].dropna().nunique()}")
if "month" in df_all_src.columns:
    months = sorted(df_all_src["month"].dropna().unique())
    print(f"Months present                            : {months}")

In [ ]:
# Drop rows without a valid detector number — not useful for focal-plane heatmaps
df_fp_valid = df_all_src.dropna(subset=["r:detector"]).copy()
df_fp_valid["r:detector"] = df_fp_valid["r:detector"].astype(int)

print(f"Rows kept for focal-plane analysis: {len(df_fp_valid):,}")
print("\nAlert count per group:")
print(df_fp_valid.groupby("group").size().sort_values(ascending=False).to_string())

## 4. Helper: draw one focal-plane heatmap on a given Axes

In [ ]:
def draw_focal_plane(
    ax,
    counts_per_detector,
    geom_df,
    norm,
    cmap_obj,
    label_detectors=True,
    label_fontsize=5,
):
    """
    Draw CCD polygons on *ax* coloured by counts_per_detector.

    Parameters
    ----------
    ax               : matplotlib Axes
    counts_per_detector : pd.Series indexed by detector number (int)
    geom_df          : geometry DataFrame (one row per CCD)
    norm             : matplotlib Normalize instance (shared across subplots)
    cmap_obj         : matplotlib colormap
    label_detectors  : if True, annotate each CCD with its detector number
    label_fontsize   : font size for detector labels
    """
    geom_local = geom_df.copy()
    geom_local["n_alerts"] = geom_local["detector"].map(counts_per_detector).fillna(0).astype(float)

    for _, row in geom_local.iterrows():
        corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
        val = row["n_alerts"]
        face_color = cmap_obj(norm(val)) if val > 0 else "lightgrey"
        poly = patches.Polygon(
            corners,
            facecolor=face_color,
            edgecolor="black",
            linewidth=0.2,
        )
        ax.add_patch(poly)
        if label_detectors:
            ax.text(
                row["x_center"],
                row["y_center"],
                str(int(row["detector"])),
                ha="center",
                va="center",
                fontsize=label_fontsize,
                fontweight="bold",
                color="k",
            )

    ax.set_aspect("equal")
    ax.set_xlim(fp_xmin, fp_xmax)
    ax.set_ylim(fp_ymin, fp_ymax)
    ax.set_xticks([])
    ax.set_yticks([])


print("draw_focal_plane() defined.")

## 5. Global focal-plane heatmap — all groups, all bands, all time

Alert count per CCD summed over all source groups, all photometric bands,
and the entire observation period.

In [ ]:
# Per-CCD alert count — all groups, all bands
counts_global = df_fp_valid.groupby("r:detector").size().rename("n_alerts")
print(f"CCDs with at least one alert: {len(counts_global)} / {len(geom)}")
print(f"Total alerts: {counts_global.sum():,}")
print(f"CCD count range: [{counts_global.min()}, {counts_global.max()}]")

vmin_g = max(counts_global.min(), 1)
vmax_g = counts_global.max()
norm_global = mcolors.LogNorm(vmin=vmin_g, vmax=vmax_g)
cmap_global = plt.get_cmap("viridis")

fig_global, ax_global = plt.subplots(figsize=(9, 9))
draw_focal_plane(
    ax_global,
    counts_global,
    geom,
    norm_global,
    cmap_global,
    label_detectors=True,
    label_fontsize=5,
)
ax_global.set_xlabel("Focal plane X [mm]")
ax_global.set_ylabel("Focal plane Y [mm]")
ax_global.set_xticks(np.linspace(fp_xmin, fp_xmax, 5))
ax_global.set_yticks(np.linspace(fp_ymin, fp_ymax, 5))
ax_global.xaxis.set_tick_params(labelbottom=True)
ax_global.yaxis.set_tick_params(labelleft=True)

sm = plt.cm.ScalarMappable(cmap=cmap_global, norm=norm_global)
sm.set_array([])
plt.colorbar(sm, ax=ax_global, fraction=0.046, pad=0.04, label="Alert count per CCD (log scale)")

ax_global.set_title(
    f"LSSTCam Focal Plane — Total alert count per CCD\n"
    f"(all groups, all bands, N_total={counts_global.sum():,})",
    fontsize=12,
)
plt.tight_layout()
savefig("focal_plane_alert_count_global")
plt.show()

## 6. Per-band focal-plane heatmaps — 2×3 grid (shared colour scale)

Alert counts are shown separately for each photometric band `u g r i z y`.
A **single log colour scale** is shared across all six subplots.

In [ ]:
# Per-band per-CCD alert counts
band_counts = {}
for band in BANDS:
    df_b = df_fp_valid[df_fp_valid["r:band"] == band]
    band_counts[band] = df_b.groupby("r:detector").size()

# Global colour range across all bands
all_vals = pd.concat(list(band_counts.values()))
global_vmin = max(int(all_vals.min()), 1)
global_vmax = int(all_vals.max())
norm_shared = mcolors.LogNorm(vmin=global_vmin, vmax=global_vmax)
cmap_shared = plt.get_cmap("viridis")

print(f"Global alert count range (all bands): [{global_vmin}, {global_vmax}]")
for band in BANDS:
    s = band_counts[band]
    if len(s) > 0:
        print(f"  band {band}: {int(s.sum()):6,} alerts | CCDs [{int(s.min())}, {int(s.max())}]")
    else:
        print(f"  band {band}: no data")

In [ ]:
fig_bands, axes_bands = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)

for idx, band in enumerate(BANDS):
    row_idx, col_idx = divmod(idx, 3)
    ax = axes_bands[row_idx, col_idx]

    draw_focal_plane(
        ax,
        band_counts[band],
        geom,
        norm_shared,
        cmap_shared,
        label_detectors=True,
        label_fontsize=5,
    )

    n_total_band = int(band_counts[band].sum()) if len(band_counts[band]) > 0 else 0
    ax.set_title(
        rf"Band {band}   (N={n_total_band:,})",
        fontsize=11,
        color=BAND_COLORS[band],
        fontweight="bold",
    )

# Single shared colour bar on the right
sm = plt.cm.ScalarMappable(cmap=cmap_shared, norm=norm_shared)
sm.set_array([])
cbar = fig_bands.colorbar(sm, ax=axes_bands, fraction=0.02, pad=0.02)
cbar.set_label("Alert count per CCD  (log scale)", fontsize=11)

fig_bands.suptitle(
    "LSSTCam Focal Plane — Alert count per band\n(all groups | shared log colour scale | grey = no alert)",
    fontsize=13,
)
savefig("focal_plane_alert_count_per_band_grid")
plt.show()

## 7. Month-by-month focal-plane heatmaps

Each subplot shows the alert count per CCD for one calendar month (YYYY-MM).
The colour scale is **independent per month** so that the spatial coverage
pattern is visible even in low-statistics months.
An optional **shared-scale variant** is also produced for direct count comparison.

Subplots are arranged in rows of `NCOLS_MONTH` columns, one panel per month.

In [ ]:
# Sorted list of months present in the data
months_present = sorted(df_fp_valid["month"].dropna().unique())
n_months = len(months_present)

print(f"Months in data: {n_months}")
for m in months_present:
    n_m = (df_fp_valid["month"] == m).sum()
    n_ccd_m = df_fp_valid[df_fp_valid["month"] == m]["r:detector"].nunique()
    print(f"  {m} : {n_m:6,} alerts  {n_ccd_m:3d} CCDs")

In [ ]:
# ── Layout parameters ─────────────────────────────────────────────────────────
NCOLS_MONTH = 4  # number of columns in the month subplot grid
n_rows_month = int(np.ceil(n_months / NCOLS_MONTH))

# Per-month per-CCD alert counts
month_counts = {}
for m in months_present:
    df_m = df_fp_valid[df_fp_valid["month"] == m]
    month_counts[m] = df_m.groupby("r:detector").size()

print(
    f"Grid: {n_rows_month} rows × {NCOLS_MONTH} cols = {n_rows_month * NCOLS_MONTH} panels (for {n_months} months)"
)

### 7a. Month grid — individual colour scale per month

Each panel uses its own log-normalisation, making spatial coverage patterns
visible even in months with few alerts.

In [ ]:
PANEL_SIZE = 3.8  # inches per subplot

fig_months_ind, axes_months_ind = plt.subplots(
    n_rows_month,
    NCOLS_MONTH,
    figsize=(PANEL_SIZE * NCOLS_MONTH, PANEL_SIZE * n_rows_month),
    constrained_layout=True,
)
# Ensure axes is always a 2D array
if n_rows_month == 1:
    axes_months_ind = axes_months_ind.reshape(1, NCOLS_MONTH)

for idx, m in enumerate(months_present):
    row_idx, col_idx = divmod(idx, NCOLS_MONTH)
    ax = axes_months_ind[row_idx, col_idx]

    counts_m = month_counts[m]
    n_total_m = int(counts_m.sum())
    n_ccd_m = len(counts_m)

    # Individual log norm for this month
    vmin_m = max(int(counts_m.min()), 1)
    vmax_m = int(counts_m.max())
    norm_m = mcolors.LogNorm(vmin=vmin_m, vmax=vmax_m)
    cmap_m = plt.get_cmap("plasma")

    draw_focal_plane(
        ax,
        counts_m,
        geom,
        norm_m,
        cmap_m,
        label_detectors=False,  # too crowded in small panels
    )

    # Add a per-panel colour bar
    sm_m = plt.cm.ScalarMappable(cmap=cmap_m, norm=norm_m)
    sm_m.set_array([])
    plt.colorbar(sm_m, ax=ax, fraction=0.046, pad=0.01)

    ax.set_title(
        f"{m}\nN={n_total_m:,}  CCDs={n_ccd_m}",
        fontsize=9,
    )

# Hide unused panels
for idx in range(n_months, n_rows_month * NCOLS_MONTH):
    row_idx, col_idx = divmod(idx, NCOLS_MONTH)
    axes_months_ind[row_idx, col_idx].set_visible(False)

fig_months_ind.suptitle(
    "LSSTCam Focal Plane — Alert count per CCD, month by month\n"
    "(all groups | individual log scale per month | grey = no alert)",
    fontsize=13,
)
savefig("focal_plane_alert_count_monthly_individual_scale")
plt.show()

### 7b. Month grid — shared colour scale

A single log-normalisation shared across all months allows direct comparison
of alert counts between periods.

In [ ]:
# Global range across all months
all_month_vals = pd.concat(list(month_counts.values()))
global_vmin_m = max(int(all_month_vals.min()), 1)
global_vmax_m = int(all_month_vals.max())
norm_months_shared = mcolors.LogNorm(vmin=global_vmin_m, vmax=global_vmax_m)
cmap_months_shared = plt.get_cmap("plasma")

print(f"Shared colour scale: [{global_vmin_m}, {global_vmax_m}] (log)")

fig_months_sh, axes_months_sh = plt.subplots(
    n_rows_month,
    NCOLS_MONTH,
    figsize=(PANEL_SIZE * NCOLS_MONTH, PANEL_SIZE * n_rows_month),
    constrained_layout=True,
)
if n_rows_month == 1:
    axes_months_sh = axes_months_sh.reshape(1, NCOLS_MONTH)

for idx, m in enumerate(months_present):
    row_idx, col_idx = divmod(idx, NCOLS_MONTH)
    ax = axes_months_sh[row_idx, col_idx]

    counts_m = month_counts[m]
    n_total_m = int(counts_m.sum())
    n_ccd_m = len(counts_m)

    draw_focal_plane(
        ax,
        counts_m,
        geom,
        norm_months_shared,
        cmap_months_shared,
        label_detectors=False,
    )

    ax.set_title(
        f"{m}\nN={n_total_m:,}  CCDs={n_ccd_m}",
        fontsize=9,
    )

# Hide unused panels
for idx in range(n_months, n_rows_month * NCOLS_MONTH):
    row_idx, col_idx = divmod(idx, NCOLS_MONTH)
    axes_months_sh[row_idx, col_idx].set_visible(False)

# Single shared colour bar
sm_sh = plt.cm.ScalarMappable(cmap=cmap_months_shared, norm=norm_months_shared)
sm_sh.set_array([])
cbar_sh = fig_months_sh.colorbar(sm_sh, ax=axes_months_sh, fraction=0.02, pad=0.02)
cbar_sh.set_label("Alert count per CCD  (log scale)", fontsize=11)

fig_months_sh.suptitle(
    "LSSTCam Focal Plane — Alert count per CCD, month by month\n"
    "(all groups | shared log scale | grey = no alert)",
    fontsize=13,
)
savefig("focal_plane_alert_count_monthly_shared_scale")
plt.show()

## 8. Monthly time series — total alert count and CCD coverage

Summary bar charts showing the evolution of:
- total number of alerts per month
- number of CCDs with at least one alert per month

In [ ]:
# Build monthly summary table
monthly_summary_rows = []
for m in months_present:
    df_m = df_fp_valid[df_fp_valid["month"] == m]
    monthly_summary_rows.append(
        {
            "month": m,
            "n_alerts": len(df_m),
            "n_ccd": df_m["r:detector"].nunique(),
            "n_visits": df_m["r:visit"].dropna().nunique() if "r:visit" in df_m.columns else 0,
            "n_objects": df_m["diaObjectId"].nunique() if "diaObjectId" in df_m.columns else 0,
        }
    )

df_monthly = pd.DataFrame(monthly_summary_rows)
print("Monthly summary:")
print(df_monthly.to_string(index=False))

# Save
csv_path = os.path.join(DIR_FIGS, "monthly_alert_summary.csv")
df_monthly.to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}")

In [ ]:
fig_ts, axes_ts = plt.subplots(2, 1, figsize=(max(8, 1.2 * n_months), 6), constrained_layout=True)

x = np.arange(n_months)
width = 0.7

# Top: total alerts per month
axes_ts[0].bar(x, df_monthly["n_alerts"], width=width, color="steelblue", edgecolor="k", linewidth=0.5)
axes_ts[0].set_xticks(x)
axes_ts[0].set_xticklabels(df_monthly["month"], rotation=45, ha="right", fontsize=9)
axes_ts[0].set_ylabel("Total alerts")
axes_ts[0].set_title("Monthly alert count — all groups", fontweight="bold")
for i, v in enumerate(df_monthly["n_alerts"]):
    axes_ts[0].text(i, v * 1.02, f"{v:,}", ha="center", fontsize=8)

# Bottom: CCDs covered per month
axes_ts[1].bar(x, df_monthly["n_ccd"], width=width, color="darkorange", edgecolor="k", linewidth=0.5)
axes_ts[1].axhline(len(geom), color="red", ls="--", lw=1, label=f"Total CCDs ({len(geom)})")
axes_ts[1].set_xticks(x)
axes_ts[1].set_xticklabels(df_monthly["month"], rotation=45, ha="right", fontsize=9)
axes_ts[1].set_ylabel("Number of CCDs with alerts")
axes_ts[1].set_title("Monthly CCD coverage", fontweight="bold")
axes_ts[1].legend(fontsize=9)
for i, v in enumerate(df_monthly["n_ccd"]):
    axes_ts[1].text(i, v * 1.02, str(v), ha="center", fontsize=8)

fig_ts.suptitle("Temporal evolution of Fink/LSST alert statistics", fontsize=12)
savefig("monthly_alert_timeseries")
plt.show()

## 9. Per-group monthly bar chart

Stacked bar chart showing the contribution of each source group to
the monthly alert count.

In [ ]:
# Pivot: rows = months, columns = groups
df_monthly_group = (
    df_fp_valid.groupby(["month", "group"]).size().unstack(fill_value=0).reindex(months_present)
)

all_groups_present = list(df_monthly_group.columns)
print(f"Groups: {all_groups_present}")
print(df_monthly_group)

In [ ]:
# Use a categorical colour map for groups
cmap_cat = plt.get_cmap("tab20")
group_colors = {g: cmap_cat(i / max(len(all_groups_present), 1)) for i, g in enumerate(all_groups_present)}

fig_stk, ax_stk = plt.subplots(figsize=(max(10, 1.5 * n_months), 5), constrained_layout=True)

bottom = np.zeros(n_months)
x = np.arange(n_months)

for grp in all_groups_present:
    vals = df_monthly_group[grp].values.astype(float)
    ax_stk.bar(
        x,
        vals,
        bottom=bottom,
        label=grp,
        color=group_colors[grp],
        edgecolor="k",
        linewidth=0.3,
        width=0.75,
    )
    bottom += vals

ax_stk.set_xticks(x)
ax_stk.set_xticklabels(months_present, rotation=45, ha="right", fontsize=9)
ax_stk.set_ylabel("Number of alerts")
ax_stk.set_title("Monthly alert count per source group (stacked)", fontweight="bold")
ax_stk.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left", ncol=1)

savefig("monthly_alert_count_per_group_stacked")
plt.show()

## 10. Per-month, per-band focal-plane heatmaps (all bands combined per month)

For each month, a 2×3 subplot grid shows the per-band alert count per CCD.
This gives a detailed view of the photometric-band coverage evolution.

One figure per month is saved to `DIR_FIGS`.

In [ ]:
# Global colour range across all months and all bands (for a shared scale within each month figure)
global_max_band_month = 1
for m in months_present:
    df_m = df_fp_valid[df_fp_valid["month"] == m]
    for band in BANDS:
        s = df_m[df_m["r:band"] == band].groupby("r:detector").size()
        if len(s) > 0:
            global_max_band_month = max(global_max_band_month, int(s.max()))

print(f"Global per-band-per-month CCD max count: {global_max_band_month}")

norm_bm = mcolors.LogNorm(vmin=1, vmax=global_max_band_month)
cmap_bm = plt.get_cmap("viridis")

for m in months_present:
    df_m = df_fp_valid[df_fp_valid["month"] == m]
    n_tot_m = len(df_m)

    fig_bm, axes_bm = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)

    for idx, band in enumerate(BANDS):
        row_idx, col_idx = divmod(idx, 3)
        ax = axes_bm[row_idx, col_idx]

        df_mb = df_m[df_m["r:band"] == band]
        counts_mb = df_mb.groupby("r:detector").size()
        n_band = int(counts_mb.sum()) if len(counts_mb) > 0 else 0

        draw_focal_plane(
            ax,
            counts_mb,
            geom,
            norm_bm,
            cmap_bm,
            label_detectors=False,
        )
        ax.set_title(
            rf"Band {band}   (N={n_band:,})",
            fontsize=11,
            color=BAND_COLORS[band],
            fontweight="bold",
        )

    # Single shared colour bar
    sm_bm = plt.cm.ScalarMappable(cmap=cmap_bm, norm=norm_bm)
    sm_bm.set_array([])
    cbar_bm = fig_bm.colorbar(sm_bm, ax=axes_bm, fraction=0.02, pad=0.02)
    cbar_bm.set_label("Alert count per CCD  (log scale)", fontsize=11)

    fig_bm.suptitle(
        f"LSSTCam Focal Plane — Alert count per band  |  {m}\n(all groups | N_total={n_tot_m:,})",
        fontsize=13,
    )
    savefig(f"focal_plane_per_band_{m}")
    plt.show()

## 11. Summary statistics table

In [ ]:
# Global summary: per-group statistics
summary_global = (
    df_fp_valid.groupby("group")
    .agg(
        n_alerts=("r:detector", "size"),
        n_ccd=("r:detector", "nunique"),
        n_visits=("r:visit", "nunique") if "r:visit" in df_fp_valid.columns else ("r:detector", lambda x: 0),
        n_objects=("diaObjectId", "nunique")
        if "diaObjectId" in df_fp_valid.columns
        else ("r:detector", lambda x: 0),
    )
    .sort_values("n_alerts", ascending=False)
    .reset_index()
)

print("Global alert statistics per source group:")
print(summary_global.to_string(index=False))

csv_global = os.path.join(DIR_FIGS, "global_alert_statistics_per_group.csv")
summary_global.to_csv(csv_global, index=False)
print(f"\nSaved: {csv_global}")